In [0]:
from pyspark.sql.functions import *

base_path = "/Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta"
silver_tables = ["transactions", "accounts", "branches", "cards", "customers"]

for table_name in silver_tables:
    silver_path = f"{base_path}/silver/{table_name}"
    gold_path = f"{base_path}/gold/{table_name}"
    df = spark.read.format("delta").load(silver_path)
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path)
    print(f"✓ {table_name}: {df.count():,} records")

print("\n✅ Silver to Gold push complete")

✓ transactions: 116,162 records
✓ accounts: 10 records
✓ branches: 4 records
✓ cards: 10 records
✓ customers: 10 records

✅ Silver to Gold push complete


### KPI Table - Business Metrics

In [0]:
from pyspark.sql.functions import *
from datetime import datetime

base_path = "/Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta"

transactions_df = spark.read.format("delta").load(f"{base_path}/gold/transactions")
accounts_df = spark.read.format("delta").load(f"{base_path}/gold/accounts")
branches_df = spark.read.format("delta").load(f"{base_path}/gold/branches")
cards_df = spark.read.format("delta").load(f"{base_path}/gold/cards")
customers_df = spark.read.format("delta").load(f"{base_path}/gold/customers")

transactions_df.display()
accounts_df.display()
branches_df.display()
cards_df.display()
customers_df.display()

account_id,transaction_date,amount,transaction_type,balance
1196428',2019-03-05,1000000.0,debit,-1.68284755405E9
1196428',2019-03-05,400000.0,debit,-1.68794226613E9
1196428',2019-03-05,300000.0,debit,-1.68722433183E9
1196428',2019-03-05,500000.0,credit,-1.68984755405E9
1196428',2019-03-05,1000000.0,debit,-1.68034755405E9
1196428',2019-03-05,568342.08,debit,-1.68654248413E9
1196428',2019-03-05,300000.0,credit,-1.68692433183E9
1196428',2019-03-05,1.0E7,credit,-1.67984755405E9
1196428',2019-03-05,1000000.0,debit,-1.68384755405E9
1196428',2019-03-05,56915.0,credit,-1.68746741683E9


account_id,latest_balance,account_type,branch_id
1196428',-1.68284755405E9,Savings,B4
1196711',-1.58691558925E9,Savings,B1
409000362497',-1.90190209261E9,Savings,B1
409000405747',-5.4826745865E8,Savings,B2
409000425051',-3.5673478865E8,Savings,B3
409000438611',-5.4821049986E8,Savings,B1
409000438620',-5.3945395387E8,Savings,B3
409000493201',743583.32,Savings,B4
409000493210',-5.4624972234E8,Savings,B2
409000611074',462200.0,Savings,B1


branch_id,branch_name,city
B1,Main Branch,Hyderabad
B2,Central Branch,Mumbai
B3,North Branch,Delhi
B4,South Branch,Chennai


card_id,account_id,card_type
C_1196428',1196428',Debit
C_409000362497',409000362497',Debit
C_409000438620',409000438620',Debit
C_409000493210',409000493210',Credit
C_409000425051',409000425051',Credit
C_1196711',1196711',Debit
C_409000438611',409000438611',Debit
C_409000493201',409000493201',Credit
C_409000405747',409000405747',Debit
C_409000611074',409000611074',Debit


customer_id,account_id,customer_name,city
1196711',1196711',Customer_1196711',Bangalore
409000362497',409000362497',Customer_409000362497',Bangalore
409000438611',409000438611',Customer_409000438611',Bangalore
409000425051',409000425051',Customer_409000425051',Delhi
409000405747',409000405747',Customer_409000405747',Hyderabad
1196428',1196428',Customer_1196428',Bangalore
409000611074',409000611074',Customer_409000611074',Bangalore
409000493201',409000493201',Customer_409000493201',Mumbai
409000438620',409000438620',Customer_409000438620',Bangalore
409000493210',409000493210',Customer_409000493210',Hyderabad


In [0]:
total_transactions = transactions_df.count()
total_transaction_amount = transactions_df.agg(sum("amount")).collect()[0][0]
avg_transaction_amount = transactions_df.agg(avg("amount")).collect()[0][0]
total_accounts = accounts_df.count()
active_accounts = accounts_df.count()
total_customers = customers_df.count()
total_branches = branches_df.count()
total_cards = cards_df.count()
active_cards = cards_df.count()

In [0]:
kpi_records = [
    ("total_transactions", float(total_transactions), "count", "Total number of transactions"),
    ("total_transaction_amount", float(total_transaction_amount), "amount", "Total transaction amount"),
    ("avg_transaction_amount", float(avg_transaction_amount), "amount", "Average transaction amount"),
    ("total_accounts", float(total_accounts), "count", "Total number of accounts"),
    ("active_accounts", float(active_accounts), "count", "Number of active accounts"),
    ("total_customers", float(total_customers), "count", "Total number of customers"),
    ("total_branches", float(total_branches), "count", "Total number of branches"),
    ("total_cards", float(total_cards), "count", "Total number of cards"),
    ("active_cards", float(active_cards), "count", "Number of active cards"),
    ("avg_accounts_per_customer", float(total_accounts / total_customers), "ratio", "Average accounts per customer"),
    ("avg_cards_per_customer", float(total_cards / total_customers), "ratio", "Average cards per customer")
]

kpi_schema = ["kpi_name", "kpi_value", "kpi_type", "kpi_description"]
kpi_df = spark.createDataFrame(kpi_records, kpi_schema)

In [0]:
kpi_path = f"{base_path}/gold/kpi_business_metrics"
kpi_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(kpi_path)
spark.read.format("delta").load(kpi_path).display()

kpi_name,kpi_value,kpi_type,kpi_description
total_transactions,116162.0,count,Total number of transactions
total_transaction_amount,4.7856984760826404E11,amount,Total transaction amount
avg_transaction_amount,4119848.552954149,amount,Average transaction amount
total_accounts,10.0,count,Total number of accounts
active_accounts,10.0,count,Number of active accounts
total_customers,10.0,count,Total number of customers
total_branches,4.0,count,Total number of branches
total_cards,10.0,count,Total number of cards
active_cards,10.0,count,Number of active cards
avg_accounts_per_customer,1.0,ratio,Average accounts per customer


In [0]:
print("Task 5: Final Report Table")

final_report_df = transactions_df \
    .join(accounts_df, transactions_df.account_id == accounts_df.account_id, "inner") \
    .join(customers_df, accounts_df.account_id == customers_df.account_id, "inner") \
    .join(branches_df, accounts_df.branch_id == branches_df.branch_id, "inner") \
    .join(cards_df, accounts_df.account_id == cards_df.account_id, "left") \
    .select(
        accounts_df.account_id,
        customers_df.customer_name,
        customers_df.city.alias("customer_city"),
        branches_df.branch_name,
        cards_df.card_type,
        transactions_df.transaction_date,
        transactions_df.amount,
        transactions_df.transaction_type,
        accounts_df.latest_balance
    )

final_report_path = f"{base_path}/gold/final_report"
final_report_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(final_report_path)

print(f"Saved: {final_report_path}")
print(f"Records: {final_report_df.count():,}")
display(final_report_df.limit(20))

Task 5: Final Report Table
Saved: /Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/gold/final_report
Records: 116,162


account_id,customer_name,customer_city,branch_name,card_type,transaction_date,amount,transaction_type,latest_balance
1196428',Customer_1196428',Bangalore,South Branch,Debit,2019-03-05,1000000.0,debit,-1.68284755405E9
1196428',Customer_1196428',Bangalore,South Branch,Debit,2019-03-05,400000.0,debit,-1.68284755405E9
1196428',Customer_1196428',Bangalore,South Branch,Debit,2019-03-05,300000.0,debit,-1.68284755405E9
1196428',Customer_1196428',Bangalore,South Branch,Debit,2019-03-05,500000.0,credit,-1.68284755405E9
1196428',Customer_1196428',Bangalore,South Branch,Debit,2019-03-05,1000000.0,debit,-1.68284755405E9
1196428',Customer_1196428',Bangalore,South Branch,Debit,2019-03-05,568342.08,debit,-1.68284755405E9
1196428',Customer_1196428',Bangalore,South Branch,Debit,2019-03-05,300000.0,credit,-1.68284755405E9
1196428',Customer_1196428',Bangalore,South Branch,Debit,2019-03-05,1.0E7,credit,-1.68284755405E9
1196428',Customer_1196428',Bangalore,South Branch,Debit,2019-03-05,1000000.0,debit,-1.68284755405E9
1196428',Customer_1196428',Bangalore,South Branch,Debit,2019-03-05,56915.0,credit,-1.68284755405E9


### AGGREGATION

In [0]:
print("Aggregation 1: Customer Transaction Behavior")

customer_behavior_df = transactions_df \
    .join(accounts_df, "account_id", "inner") \
    .join(customers_df, "account_id", "inner") \
    .groupBy(
        customers_df.customer_id,
        customers_df.customer_name,
        customers_df.city
    ) \
    .agg(
        count("*").alias("total_transactions"),
        sum("amount").alias("total_spent"),
        avg("amount").alias("avg_transaction_amount"),
        min("amount").alias("min_transaction"),
        max("amount").alias("max_transaction"),
        max("transaction_date").alias("last_transaction_date")
    ) \
    .withColumn("customer_segment",
        when(col("total_spent") > 50000000, "Premium")
        .when(col("total_spent") > 20000000, "Gold")
        .when(col("total_spent") > 5000000, "Silver")
        .otherwise("Standard")
    ) \
    .orderBy(col("total_spent").desc())

customer_behavior_path = f"{base_path}/gold/agg_customer_behavior"
customer_behavior_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(customer_behavior_path)

print(f"Saved: {customer_behavior_path}")
display(customer_behavior_df.limit(20))

Aggregation 1: Customer Transaction Behavior
Saved: /Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/gold/agg_customer_behavior


customer_id,customer_name,city,total_transactions,total_spent,avg_transaction_amount,min_transaction,max_transaction,last_transaction_date,customer_segment
409000362497',Customer_409000362497',Bangalore,29840,2.036560568172809E11,6824934.879935687,0.03,2.0E8,2019-03-05,Premium
1196428',Customer_1196428',Bangalore,48758,1.3675033977710992E11,2804674.920569136,0.25,2.119594422E8,2019-03-05,Premium
1196711',Customer_1196711',Bangalore,10518,9.260772833735988E10,8804689.897067873,0.25,5.0E8,2019-02-28,Premium
409000438620',Customer_409000438620',Bangalore,13454,3.4398517966489975E10,2556750.2576549705,0.07,5.448E8,2019-03-05,Premium
409000438611',Customer_409000438611',Bangalore,4588,9.411450687859995E9,2051318.807292937,0.03,2.4E8,2019-03-05,Premium
409000405747',Customer_409000405747',Hyderabad,51,6.491031024E8,1.2727511811764706E7,21.0,2.021E8,2019-03-02,Premium
409000425051',Customer_409000425051',Delhi,802,4.1154209903E8,513144.761882793,1.0,3.54E8,2019-03-02,Premium
409000611074',Customer_409000611074',Bangalore,1093,2.91257038E8,266474.8746569076,120.0,3000000.0,2019-02-08,Premium
409000493210',Customer_409000493210',Hyderabad,6014,2.0281453757000002E8,33723.73421516462,0.01,1.5E7,2019-03-05,Premium
409000493201',Customer_409000493201',Mumbai,1044,1.9103724516000006E8,182985.8670114943,0.9,2500000.0,2019-03-05,Premium


In [0]:
print("Task 6: Ranking Top Accounts by Balance")

from pyspark.sql.window import Window

window_spec = Window.orderBy(col("latest_balance").desc())

ranked_accounts_df = accounts_df \
    .join(customers_df, accounts_df.account_id == customers_df.account_id, "inner") \
    .select(
        accounts_df.account_id,
        customers_df.customer_name,
        accounts_df.latest_balance,
        accounts_df.account_type,
        accounts_df.branch_id
    ) \
    .withColumn("balance_rank", dense_rank().over(window_spec)) \
    .withColumn("balance_row_number", row_number().over(window_spec)) \
    .orderBy("balance_rank")

ranked_accounts_path = f"{base_path}/gold/ranked_accounts"
ranked_accounts_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ranked_accounts_path)

print(f"Saved: {ranked_accounts_path}")
display(ranked_accounts_df.limit(10))

Task 6: Ranking Top Accounts by Balance


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Saved: /Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/gold/ranked_accounts


account_id,customer_name,latest_balance,account_type,branch_id,balance_rank,balance_row_number
409000493201',Customer_409000493201',743583.32,Savings,B4,1,1
409000611074',Customer_409000611074',462200.0,Savings,B1,2,2
409000425051',Customer_409000425051',-3.5673478865E8,Savings,B3,3,3
409000438620',Customer_409000438620',-5.3945395387E8,Savings,B3,4,4
409000493210',Customer_409000493210',-5.4624972234E8,Savings,B2,5,5
409000438611',Customer_409000438611',-5.4821049986E8,Savings,B1,6,6
409000405747',Customer_409000405747',-5.4826745865E8,Savings,B2,7,7
1196711',Customer_1196711',-1.58691558925E9,Savings,B1,8,8
1196428',Customer_1196428',-1.68284755405E9,Savings,B4,9,9
409000362497',Customer_409000362497',-1.90190209261E9,Savings,B1,10,10


In [0]:
print("Task 7: Running Total per Account")

from pyspark.sql.window import Window

window_spec_running = Window.partitionBy("account_id").orderBy("transaction_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_total_df = transactions_df \
    .join(accounts_df, transactions_df.account_id == accounts_df.account_id, "inner") \
    .join(customers_df, accounts_df.account_id == customers_df.account_id, "inner") \
    .select(
        transactions_df.account_id,
        customers_df.customer_name,
        transactions_df.transaction_date,
        transactions_df.amount,
        transactions_df.transaction_type
    ) \
    .withColumn("running_total", sum("amount").over(window_spec_running)) \
    .withColumn("transaction_count", row_number().over(window_spec_running)) \
    .orderBy("account_id", "transaction_date")

running_total_path = f"{base_path}/gold/running_total_transactions"
running_total_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(running_total_path)

print(f"Saved: {running_total_path}")
display(running_total_df.filter(col("account_id") == running_total_df.select("account_id").first()[0]).limit(20))

Task 7: Running Total per Account
Saved: /Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/gold/running_total_transactions


account_id,customer_name,transaction_date,amount,transaction_type,running_total,transaction_count
1196428',Customer_1196428',2015-01-01,1200000.0,credit,1200000.0,1
1196428',Customer_1196428',2015-01-01,800000.0,credit,2000000.0,2
1196428',Customer_1196428',2015-01-02,25000.0,credit,3712620.0,6
1196428',Customer_1196428',2015-01-02,1500000.0,credit,3672620.0,4
1196428',Customer_1196428',2015-01-02,25000.0,credit,9962676.18,11
1196428',Customer_1196428',2015-01-02,15000.0,credit,3687620.0,5
1196428',Customer_1196428',2015-01-02,172620.0,credit,2172620.0,3
1196428',Customer_1196428',2015-01-02,700000.0,credit,9937676.18,10
1196428',Customer_1196428',2015-01-02,5500000.0,debit,9237676.18,9
1196428',Customer_1196428',2015-01-02,56.18,debit,3737676.18,8


In [0]:
print("Task 8: Group By Analytics\n")

# 1. Total amount by transaction_type
print("1. Amount by Transaction Type")
amount_by_type_df = transactions_df \
    .groupBy("transaction_type") \
    .agg(
        count("*").alias("total_transactions"),
        sum("amount").alias("total_amount"),
        avg("amount").alias("avg_amount"),
        min("amount").alias("min_amount"),
        max("amount").alias("max_amount")
    ) \
    .orderBy(col("total_amount").desc())

amount_by_type_path = f"{base_path}/gold/agg_amount_by_transaction_type"
amount_by_type_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(amount_by_type_path)
print(f"Saved: {amount_by_type_path}")
display(amount_by_type_df)

# 2. Transactions per branch
print("\n2. Transactions per Branch")
transactions_by_branch_df = transactions_df \
    .join(accounts_df, "account_id", "inner") \
    .join(branches_df, accounts_df.branch_id == branches_df.branch_id, "inner") \
    .groupBy(
        branches_df.branch_id,
        branches_df.branch_name,
        branches_df.city
    ) \
    .agg(
        count("*").alias("total_transactions"),
        sum("amount").alias("total_amount"),
        countDistinct(transactions_df.account_id).alias("unique_accounts")
    ) \
    .orderBy(col("total_transactions").desc())

transactions_by_branch_path = f"{base_path}/gold/agg_transactions_by_branch"
transactions_by_branch_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(transactions_by_branch_path)
print(f"Saved: {transactions_by_branch_path}")
display(transactions_by_branch_df)

# 3. Customers per city
print("\n3. Customers per City")
customers_by_city_df = customers_df \
    .join(accounts_df, customers_df.account_id == accounts_df.account_id, "inner") \
    .groupBy(customers_df.city) \
    .agg(
        countDistinct(customers_df.customer_id).alias("total_customers"),
        count(accounts_df.account_id).alias("total_accounts"),
        avg(accounts_df.latest_balance).alias("avg_balance")
    ) \
    .withColumn("accounts_per_customer",
        round(col("total_accounts") / col("total_customers"), 2)
    ) \
    .orderBy(col("total_customers").desc())

customers_by_city_path = f"{base_path}/gold/agg_customers_by_city"
customers_by_city_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(customers_by_city_path)
print(f"Saved: {customers_by_city_path}")
display(customers_by_city_df)

print("\n✅ All analytics completed")

Task 8: Group By Analytics

1. Amount by Transaction Type
Saved: /Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/gold/agg_amount_by_transaction_type


transaction_type,total_transactions,total_amount,avg_amount,min_amount,max_amount
debit,53525,2.402011322847603E11,4487643.760574691,0.01,4.5944754636E8
credit,62637,2.383687153235002E11,3805557.6627791915,0.01,5.448E8



2. Transactions per Branch
Saved: /Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/gold/agg_transactions_by_branch


branch_id,branch_name,city,total_transactions,total_amount,unique_accounts
B4,South Branch,Chennai,49802,1.3694137702226993E11,2
B1,Main Branch,Hyderabad,46039,3.059664928805008E11,4
B3,North Branch,Delhi,14256,3.481006006551997E10,2
B2,Central Branch,Mumbai,6065,8.5191763997E8,2



3. Customers per City
Saved: /Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/gold/agg_customers_by_city


city,total_customers,total_accounts,avg_balance,accounts_per_customer
Bangalore,6,6,-1.0431445816066666E9,1.0
Hyderabad,2,2,-5.47258590495E8,1.0
Delhi,1,1,-3.5673478865E8,1.0
Mumbai,1,1,743583.32,1.0



✅ All analytics completed
